## Tutorial 9, Question 1

This code demonstrates an important limitation of self-attention:

Without positional encoding, self-attention knows which tokens are present, but it does not know their order.

We test this by rearranging the input tokens and observing how the outputs change. We then add sinusoidal positional encoding and repeat the experiment.

In [7]:
import math
import torch
import torch.nn as nn

## 1. Sinusoidal positional encoding

For a token at position $pos$, the positional encoding is defined as

$$
PE(pos,2i)
=
\sin\left(
\frac{pos}{10000^{2i/D}}
\right),
$$

$$
PE(pos,2i+1)
=
\cos\left(
\frac{pos}{10000^{2i/D}}
\right).
$$

Here:

- $N$ is the number of token positions.
- $D$ is the embedding dimension.
- $pos$ is the token position.
- $i$ determines the frequency used by each pair of embedding dimensions.

Even-numbered dimensions use sine, while odd-numbered dimensions use cosine. Different dimensions use different frequencies, allowing the model to represent both short-range and long-range positional relationships.

In [8]:
def sinusoidal_encoding(N, D):
    """
    Create sinusoidal positional encodings.

    Parameters
    ----------
    N : int
        Number of token positions.
    D : int
        Embedding dimension. D should be even.

    Returns
    -------
    PE : torch.Tensor
        Positional encoding matrix with shape [N, D].
    """

    # Position indices: 0, 1, ..., N-1
    # Shape: [N, 1]
    positions = torch.arange(
        N, dtype=torch.float32
    ).unsqueeze(1)

    # One frequency for each sine-cosine pair
    # Shape: [D/2]
    frequencies = torch.exp(
        torch.arange(0, D, 2, dtype=torch.float32)
        * (-math.log(10000.0) / D)
    )

    # Initialize the positional encoding matrix
    # Shape: [N, D]
    PE = torch.zeros(N, D)

    # Put sine values in dimensions 0, 2, 4, ...
    PE[:, 0::2] = torch.sin(positions * frequencies)

    # Put cosine values in dimensions 1, 3, 5, ...
    PE[:, 1::2] = torch.cos(positions * frequencies)

    return PE

### Understanding the tensor shapes

For $N=4$ and $D=8$:

- `positions` has shape `[4, 1]`.
- `frequencies` has shape `[4]`.
- `positions * frequencies` uses broadcasting to produce a tensor of shape `[4, 4]`.
- The final positional encoding has shape `[4, 8]`.

The four frequencies correspond to four sine–cosine pairs:

| Dimensions | Frequency |
|---|---:|
| 0 and 1 | $1$ |
| 2 and 3 | $0.1$ |
| 4 and 5 | $0.01$ |
| 6 and 7 | $0.001$ |

In [9]:
PE_example = sinusoidal_encoding(N=4, D=8)

print("Positional encoding:")
print(PE_example)
print("\nShape:", PE_example.shape)

Positional encoding:
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])

Shape: torch.Size([4, 8])


## 2. Create the input and self-attention layer

The input tensor has shape

$$
[\text{batch size},\ \text{sequence length},\ \text{embedding dimension}]
=
[1,4,8].
$$

It contains one sequence, four tokens, and an eight-dimensional embedding for each token.

The multi-head attention layer has two attention heads. Because the total embedding dimension is eight, each head processes four dimensions:

$$
D_{\text{head}} = \frac{8}{2}=4.
$$

Setting `dropout=0.0` and using evaluation mode makes the results deterministic.

In [10]:
torch.manual_seed(1)

# One sequence containing four token embeddings
X = torch.randn(1, 4, 8)

attention = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    dropout=0.0,
    batch_first=True
)

attention.eval()

print("Input shape:", X.shape)
print("Input:")
print(X)

Input shape: torch.Size([1, 4, 8])
Input:
tensor([[[-1.5256, -0.7502, -0.6540, -1.6095, -0.1002, -0.6092, -0.9798,
          -1.6091],
         [-0.7121,  0.3037, -0.7773, -0.2515, -0.2223,  1.6871,  0.2284,
           0.4676],
         [-0.6970, -1.1608,  0.6995,  0.1991,  0.8657,  0.2444, -0.6629,
           0.8073],
         [ 1.1017, -0.1759, -2.2456, -1.4465,  0.0612, -0.6177, -0.7981,
          -0.1316]]])


## 3. Rearrange the input tokens

We change the token order from

$$
[0,1,2,3]
$$

to

$$
[2,0,3,1].
$$

Only the order changes. The token vectors themselves remain unchanged.

The expression `X[:, permutation]` keeps all items in the batch but rearranges the sequence dimension.

In [13]:
permutation = torch.tensor([2, 0, 3, 1])

X_permuted = X[:, permutation]

print("Original order:   [0, 1, 2, 3]")
print("Permuted order:  ", permutation.tolist())
print("Permuted input shape:", X_permuted.shape)
print("Permuted input:")
print(X_permuted)

Original order:   [0, 1, 2, 3]
Permuted order:   [2, 0, 3, 1]
Permuted input shape: torch.Size([1, 4, 8])
Permuted input:
tensor([[[-0.6970, -1.1608,  0.6995,  0.1991,  0.8657,  0.2444, -0.6629,
           0.8073],
         [-1.5256, -0.7502, -0.6540, -1.6095, -0.1002, -0.6092, -0.9798,
          -1.6091],
         [ 1.1017, -0.1759, -2.2456, -1.4465,  0.0612, -0.6177, -0.7981,
          -0.1316],
         [-0.7121,  0.3037, -0.7773, -0.2515, -0.2223,  1.6871,  0.2284,
           0.4676]]])


## 4. Test self-attention without positional encoding

The same input is used as the query, key, and value. Therefore, this is self-attention:

$$
Q=XW_Q,\qquad
K=XW_K,\qquad
V=XW_V.
$$

We apply the same attention layer to:

1. The original input sequence.
2. The permuted input sequence.

We then compare the output of the permuted sequence with the original output rearranged using the same permutation.

The error is

$$
\max\left|
Y_{\text{permuted}}-\operatorname{Permute}(Y)
\right|.
$$

If the error is approximately zero, rearranging the input merely rearranges the output in the same way.

In [14]:
# Self-attention for the original sequence
Y, _ = attention(
    query=X,
    key=X,
    value=X,
    need_weights=False
)

# Self-attention for the permuted sequence
Y_permuted, _ = attention(
    query=X_permuted,
    key=X_permuted,
    value=X_permuted,
    need_weights=False
)

# Compare the two results
error_without_PE = (
    Y_permuted - Y[:, permutation]
).abs().max()

print("Output shape:", Y.shape)
print("Error without positional encoding:",
      error_without_PE.item())

Output shape: torch.Size([1, 4, 8])
Error without positional encoding: 2.9802322387695312e-08


### Interpretation

The error should be approximately zero, apart from small floating-point rounding errors.

This demonstrates that self-attention without positional encoding is **permutation-equivariant**:

$$
\operatorname{Attention}(PX)
=
P\operatorname{Attention}(X),
$$

where $P$ represents a permutation.

In other words, rearranging the input tokens simply rearranges the outputs in the same way. Self-attention can model relationships among token contents, but it cannot determine their sequence order.

## 5. Add positional encoding

The positional encoding initially has shape `[4, 8]`. We use `unsqueeze(0)` to add a batch dimension, producing shape `[1, 4, 8]`.

It can then be added to the input:

$$
\text{input representation}
=
\text{token embedding}
+
\text{positional encoding}.
$$

The token embedding represents **what** the token is, while the positional encoding represents **where** it occurs.

In [15]:
PE = sinusoidal_encoding(
    N=4,
    D=8
).unsqueeze(0)

print("Input shape:", X.shape)
print("Positional encoding shape:", PE.shape)
print("Combined input shape:", (X + PE).shape)

Input shape: torch.Size([1, 4, 8])
Positional encoding shape: torch.Size([1, 4, 8])
Combined input shape: torch.Size([1, 4, 8])


## 6. Repeat the permutation test

For the original sequence, we calculate attention using `X + PE`.

For the permuted sequence, we calculate attention using `X_permuted + PE`.

Notice that `PE` is not permuted. This is intentional. After the tokens are rearranged, each token occupies a new position and receives the positional encoding associated with that new position.

For example, token 2 originally receives the encoding for position 2. After permutation, it becomes the first token and receives the encoding for position 0.

In [16]:
# Original sequence with positional encoding
Y_with_PE, _ = attention(
    query=X + PE,
    key=X + PE,
    value=X + PE,
    need_weights=False
)

# Permuted sequence with positional encoding
# PE is unchanged because it represents the new positions
Y_permuted_with_PE, _ = attention(
    query=X_permuted + PE,
    key=X_permuted + PE,
    value=X_permuted + PE,
    need_weights=False
)

error_with_PE = (
    Y_permuted_with_PE
    - Y_with_PE[:, permutation]
).abs().max()

print("Error without positional encoding:",
      error_without_PE.item())

print("Error with positional encoding:",
      error_with_PE.item())

Error without positional encoding: 2.9802322387695312e-08
Error with positional encoding: 0.23139895498752594


## Expected observations

The expected pattern is:

```text
Error without positional encoding: approximately 0
Error with positional encoding:    clearly greater than 0

Once positional information is included, moving a token changes the positional vector attached to it. The attention layer can therefore distinguish different token orders.

Self-attention alone can model relationships between token contents, but it does not know the order of the tokens. Positional encoding introduces sequence-order information, allowing the Transformer to distinguish sequences that contain the same tokens in different orders.